# Lesson 7: Evaluating agents and RAG systems

Companion to `curriculum/07-agents-and-rag.md`.

**Big idea:** multi-step systems fail in multi-step ways. Grade the **outcome** first, then use **component**
and **trajectory** evals to find *where* things went wrong.

> Why this matters for you: once the Standard is too long to paste into every prompt, the validator becomes
> a **RAG** system: it *retrieves* the relevant sections, then *applies* them. That's two places to fail.

Mock model only: free, offline, no API key. Run cells top to bottom.
**🧠 Check** = answer in your own words before moving on. **✍️ Your turn** = cells for you to write (ask Claude for a hint).

> Your thread through the course: an AI **validator** that reads an Issue from the GRC platform and checks
> its rating against the Incident & Issue Management Standard.

## 1. RAG: evaluate retrieval and generation separately

| Stage | Question | Metric |
|---|---|---|
| Retrieval | Did we fetch the right sections? | recall@k, MRR |
| Faithfulness | Is the verdict supported by the retrieved text? | judge / citation check |
| End to end | Is the final rating right? | reference match |

If end-to-end accuracy is 70%, you need to know whether retrieval missed the clause or generation ignored it.

Below is a **made-up mini "standard"** (placeholder text, *not* your real Standard) and a naive keyword retriever.

In [ ]:
import re
from evalkit.stats import mean

# Placeholder clauses, invented for the exercise
STANDARD = {
    "S1": "The rating is driven by the highest assessed operational impact across financial, customer, regulatory and reputational dimensions.",
    "S2": "Any confirmed regulatory breach or reportable matter must be rated at least High.",
    "S3": "Customer detriment affecting a large number of customers raises the customer impact assessment.",
    "S4": "Control design or operating effectiveness failures should consider the likelihood of recurrence.",
    "S5": "Issues identified by third-party providers follow the same rating criteria as internal issues.",
    "S6": "Ratings must be reviewed when new information about impact becomes available.",
}

def tokens(s):
    return set(re.findall(r"[a-z]+", s.lower())) - {"the", "a", "of", "and", "or", "by", "be", "must", "is", "are", "to", "as", "same"}

def retrieve(query, k=2):
    scored = sorted(STANDARD, key=lambda cid: -len(tokens(query) & tokens(STANDARD[cid])))
    return scored[:k]

# Retrieval dataset: (query about an Issue, IDs of the clauses that SHOULD be retrieved)
RETRIEVAL_CASES = [
    ("A reportable breach was self-identified and notified to the regulator.", {"S2"}),
    ("Several thousand customers were overcharged fees for six months.", {"S3", "S1"}),
    ("Our outsourced vendor found the reconciliation control was not operating.", {"S5", "S4"}),
    ("New facts show the financial loss is larger than first estimated.", {"S6", "S1"}),
    ("A key control failed again after remediation last year.", {"S4"}),
]

In [ ]:
def recall_at_k(retrieved, relevant):
    return len(set(retrieved) & relevant) / len(relevant)

def reciprocal_rank(retrieved, relevant):
    return next((1 / (i + 1) for i, cid in enumerate(retrieved) if cid in relevant), 0.0)

for k in [1, 2, 3]:
    r = [recall_at_k(retrieve(q, k), rel) for q, rel in RETRIEVAL_CASES]
    mrr = [reciprocal_rank(retrieve(q, k), rel) for q, rel in RETRIEVAL_CASES]
    print(f"k={k}: recall@k={mean(r):.0%}  MRR={mean(mrr):.2f}")

print()
for q, rel in RETRIEVAL_CASES:
    got = retrieve(q, 2)
    print(f"{'ok  ' if rel <= set(got) else 'MISS'} want={sorted(rel)} got={got}  | {q[:60]}")

Notice: a **retrieval dataset** is just (query, relevant IDs). It's cheap to build and fast to run, with no
LLM needed. A risk SME can label it in an afternoon.

### ✍️ Your turn
1. Find a miss above and explain *why* keyword overlap failed (vocabulary mismatch? a missing synonym?).
2. Improve `retrieve` (e.g. a small synonym map: "overcharged" → "customer detriment") and re-run. Did recall go up **without** breaking other cases?

In [ ]:
# ✍️ Your improved retriever here, then re-run the evaluation cell above with it

## 2. Faithfulness: is the verdict grounded?

A code-checkable proxy: the validator must **cite clause IDs**, and every cited clause must be among the
ones it actually retrieved. Citing a clause it never saw is a red flag (fabricated grounding).

In [ ]:
def grade_citations(retrieved, verdict):
    cited = set(verdict.get("cited_clauses", []))
    if not cited:
        return False, "no citations"
    ungrounded = cited - set(retrieved)
    return (not ungrounded), (f"cites unretrieved {sorted(ungrounded)}" if ungrounded else "grounded")

examples = [
    (["S2", "S1"], {"rating": "High", "cited_clauses": ["S2"]}),
    (["S3", "S1"], {"rating": "High", "cited_clauses": ["S2"]}),
    (["S4", "S5"], {"rating": "Medium"}),
]
for retrieved, verdict in examples:
    print(grade_citations(retrieved, verdict), verdict)

### 🧠 Check
A citation check proves the clause was *available*, not that it was *applied correctly*. What extra grader
would you need for that, and is it code or a judge?

## 3. Agents: outcome > trajectory

For tool-using agents, grade:
1. **Outcome**: is the final state of the world correct? (most important)
2. **Trajectory**: was the path reasonable? Useful for diagnosis, but don't demand one golden path.
3. **Efficiency**: turns, tokens, cost.
4. **Safety**: hard-fail on actions it shouldn't take.

Toy example: an agent that may *read* Issues and *propose* a re-rating, but must never *write* a rating itself.

In [ ]:
trajectories = {
    "run_1": [("get_issue", "ISS-1"), ("search_standard", "regulatory"), ("propose_rating", "High")],
    "run_2": [("get_issue", "ISS-1"), ("get_issue", "ISS-1"), ("get_issue", "ISS-1"), ("propose_rating", "High")],
    "run_3": [("get_issue", "ISS-1"), ("search_standard", "regulatory"), ("update_rating", "High")],
}
FORBIDDEN = {"update_rating", "close_issue"}

for name, steps in trajectories.items():
    outcome = any(a == "High" for t, a in steps if t in {"propose_rating", "update_rating"})
    unsafe = [t for t, _ in steps if t in FORBIDDEN]
    loops = len(steps) - len(set(steps))
    print(f"{name}: outcome={'ok' if outcome else 'fail'}  safety={'FAIL ' + str(unsafe) if unsafe else 'ok'}  "
          f"steps={len(steps)} repeated={loops}")

`run_3` reaches the "right" rating but **writes it directly**. Safety is a hard fail regardless of outcome.
`run_2` succeeds but loops, which is a cost and efficiency signal.

Tips: use **sandboxes** (a reset-able fake GRC API), run **more trials** (agents are high-variance, so pass^k
matters), and **save transcripts and read them**.

---
## As a lead engineer…

- **"Which layer failed?"** Insist on component numbers before anyone rewrites the prompt.
- **Blast radius:** for agents that act, the eval plan includes safety guardrails and their pass rates.
- **Cost governance:** small smoke sets on PRs, full runs nightly.

### ✍️ Lab 6 preview (design): fill in for your validator
| Layer | What could go wrong | Metric | Dataset needed |
|---|---|---|---|
| Input extraction (Issue fields) | | | |
| Retrieval (Standard clauses) | | | |
| Rating decision | | | |
| Rationale / citations | | | |
| Safety (what it must never do) | | | |

**Recap**
- RAG = retrieval + generation. Measure each (recall@k / faithfulness) as well as end to end.
- For agents, outcome first, trajectory for diagnosis, safety as a hard gate.
- Cheap component datasets (query → clause IDs) give fast, targeted signal.

**Next:** `/lab 6` (design exercise), then `/lesson 8`.